In [1]:
import ee 
from RadGEEToolbox import GenericCollection, LandsatCollection, get_palette
# import GEE_UBM
from GEE_UBM import InputCollections, build_model_ready_collection, OriginalUBMRun, ModifiedUBM1Run, check_merged_collection
import geemap
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd
import os

In [2]:
service_account = 'localpythonscripts@ut-gee-ugs-bsf-dev.iam.gserviceaccount.com'
credentials = ee.ServiceAccountCredentials(service_account, 'C:\\Users\\mradwin\\ut-gee-ugs-bsf-dev-53dcc5d729e0.json')
ee.Initialize(credentials=credentials)

In [3]:
UT_boundary = ee.FeatureCollection("projects/ut-gee-ugs-bsf-dev/assets/Utah_Regional_Boundary").geometry()
dirty_devil_roi = ee.FeatureCollection('projects/ut-gee-ugs-bsf-dev/assets/dirty_devil_gage_drainage_area').geometry()
duschesne_roi = ee.FeatureCollection('projects/ut-gee-ugs-bsf-dev/assets/duschesne_gage_drainage_area').geometry()
price_roi = ee.FeatureCollection('projects/ut-gee-ugs-bsf-dev/assets/price_river_gage_drainage_area').geometry()
sevier_roi = ee.FeatureCollection('projects/ut-gee-ugs-bsf-dev/assets/sevier_river_drainage_area').geometry()
uinta_roi = ee.FeatureCollection('projects/ut-gee-ugs-bsf-dev/assets/uinta_river_drainage_area').geometry()
weber_roi = ee.FeatureCollection('projects/ut-gee-ugs-bsf-dev/assets/weber_river_oakley_drainage_area').geometry()

Using the DEM to calculate TPI and ruggedness rasters

In [4]:
dem = ee.Image('USGS/SRTMGL1_003') #.clip(UT_boundary).rename('elevation')
dem_proj = dem.projection() #.mean()
dem = dem.multiply(3.28084).clip(UT_boundary).setDefaultProjection(dem_proj).rename('elevation')
# dem = dem.multiply(3.28084).clip(UT_boundary).reproject(crs='EPSG:32612', scale=30).rename('elevation')
# dem = dem.clip(UT_boundary).reproject(crs='EPSG:32612', scale=30).rename('elevation')


slope = ee.Terrain.slope(dem).setDefaultProjection(dem_proj)
# print(dem.projection().nominalScale().getInfo())

aspect = ee.Terrain.aspect(dem).setDefaultProjection(dem_proj)
# print(dem.projection().nominalScale().getInfo())

# Calculate TPI (Topographic Position Index)
# (Subtracts pixel elevation from the average of its neighbors)
# A large neighborhood (e.g., 20 pixels) helps identify valleys vs ridges.
# tpi = dem.subtract(dem.focal_mean(20, 'circle', 'pixels')).rename('TPI').setDefaultProjection(dem_proj)
# tpi = dem.subtract(dem.focal_mean(20, 'circle', 'pixels')).rename('TPI') #.setDefaultProjection(crs='EPSG:32612', scale=30)
tpi = ee.Image('projects/ut-gee-ugs-bsf-dev/assets/Utah_TPI_SRTM_30m')
print(tpi.projection().nominalScale().getInfo())

ruggedness = tpi.abs().rename('Ruggedness').setDefaultProjection(dem_proj)
lithology = ee.Image('projects/ut-gee-ugs-bsf-dev/assets/Utah_Lithology_Rasters/Utah_Lithology_USGS_RockTypes').clip(UT_boundary).rename('lithology')
land_cover = ee.ImageCollection('USGS/NLCD_RELEASES/2021_REL/NLCD').first()\
                                        .select('landcover').clip(UT_boundary)
geomaterials = ee.FeatureCollection('projects/ut-gee-ugs-bsf-dev/assets/GeoMaterial_USGS_CNGM_Utah')

30


In [5]:
previous_best_geok = InputCollections(start_date='2023-01-01', end_date='2023-12-31').get_static_raster('POLARIS_K_Sat_monthly_scaled')

In [ ]:
# k_values = {
#     'Coarse-grained, mafic-composition intrusive igneous rock': 1e-7*m_s_to_m_month_conversion,
#     'Alluvial sediment': 1e-4*m_s_to_m_month_conversion,
#     'Peat and muck': 1e-5*m_s_to_m_month_conversion,
#     'Playa sediment': 1e-8*m_s_to_m_month_conversion,
#     'Sedimentary material': 1e-6*m_s_to_m_month_conversion, #v7: 1e-7
#     'Medium and high-grade regional metamorphic rock, of unspecified origin': 1e-8*m_s_to_m_month_conversion,
#     'Sedimentary rock': 5e-7*m_s_to_m_month_conversion, #v11: 5e-7, #v10: 1e-7, #v9: 1e-10 #v8: 1e-9, #v7: 1e-8
#     'Metamorphic rock': 1e-9*m_s_to_m_month_conversion,
#     'Clastic sedimentary rock': 1e-6*m_s_to_m_month_conversion,
#     'Mostly sandstone': 5e-7*m_s_to_m_month_conversion,
#     'Clastic sediment': 1e-5*m_s_to_m_month_conversion,
#     'Debris flows, landslides, and other localized mass-movement sediment': 1e-5*m_s_to_m_month_conversion,
#     'Eolian sediment': 1e-4*m_s_to_m_month_conversion,
#     'Extrusive igneous material': 5e-6*m_s_to_m_month_conversion, #v10 and v11: 5e-6, #v9: 8e-6 #v8: 2e-5, #v7: 1e-8, #1e-7
#     'Felsic-composition pyroclastic flows': 1e-6*m_s_to_m_month_conversion,
#     'Glacial till': 0.5e-5*m_s_to_m_month_conversion, #v10 and v11: 0.5e-5, #v9: 2.5e-5, #v8: 5e-5, #v7: 1e-5, #1e-6
#     'Coarse-grained intrusive igneous rock': 1e-7*m_s_to_m_month_conversion,
#     'Coarse-grained, felsic-composition intrusive igneous rock': 1e-7*m_s_to_m_month_conversion,
#     'Intermediate-composition lava flows': 1e-5*m_s_to_m_month_conversion,
#     'Intrusive igneous rock': 1e-8*m_s_to_m_month_conversion, #1e-7
#     'Lacustrine sediment': 5e-8*m_s_to_m_month_conversion,
#     'Mafic-composition lava flows': 1e-5*m_s_to_m_month_conversion,
#     'Metasedimentary rock': 0.1e-7*m_s_to_m_month_conversion, #v11: 0.1e-7, #v10: 0.5e-7, #v9:3e-7 #v8: 5e-7, #v7: 1e-7, #1e-8
#     'Sandstone and mudstone': 5e-10*m_s_to_m_month_conversion, #v11: 5e-10, #v10: 1e-9
#     'Mostly carbonate rock': 1e-5*m_s_to_m_month_conversion,
#     'Sedimentary and extrusive igneous material': 1e-6*m_s_to_m_month_conversion,
#     'Water or ice': 1e-9*m_s_to_m_month_conversion
# }

In [ ]:
m_s_to_m_month_conversion = 60 * 60 * 24 * 30.44  # seconds to average month

k_scalar = 1
k_values = {
    'Glacial till': 1.9e-5 * m_s_to_m_month_conversion * k_scalar, #v17: 2.4e-5, #v16: 2.0e-5, #v15:2.3e-5, #v14: 1.8e-5 #v13: 1.8e-5, #v12: 2.5e-5
    'Metasedimentary rock': 1.8e-5 * m_s_to_m_month_conversion * k_scalar, #v17: 1.5e-5, #v16: 1.8e-5, #v14: 1.5e-5,#v13:8e-6   #v12: 3e-7

    'Extrusive igneous material': 0.9e-5 * m_s_to_m_month_conversion * k_scalar, #v13: 1.2e-5, #v12 8e-6 or 0.8e-5
    'Intermediate-composition lava flows': 2e-5 * m_s_to_m_month_conversion * k_scalar,
    'Mafic-composition lava flows': 1e-5 * m_s_to_m_month_conversion * k_scalar,

    'Sedimentary rock': 3.7e-6 * m_s_to_m_month_conversion * k_scalar, #v16/17: 4e-6, #v15: 2e-6, #v14: 5e-6, #v13: 1e-6    #v12: 1e-10
    'Sandstone and mudstone': 3.8e-6 * m_s_to_m_month_conversion * k_scalar, #v17: 3.5e-6, #v16: 4e-6, #v15: 2e-6, #v14: 5e-6, #v13: 5e-7, #v12: 1e-9
    'Clastic sedimentary rock': 6.5e-6 * m_s_to_m_month_conversion * k_scalar, #v17: 6e-6, #v16: 7e-6, #v15: 4e-6, #v14: 1e-5 #v13: 2e-6, #v12: 1e-6
    'Alluvial sediment': 2e-5 * m_s_to_m_month_conversion * k_scalar, #v17: 3e-5, #v16: 1e-5, #v13: 4e-5, #v12 1e-4

    'Medium and high-grade regional metamorphic rock, of unspecified origin': 2.65e-5 * m_s_to_m_month_conversion * k_scalar, #v16: 2.75e-5, #v15: 3.5e-5, #v14: 2e-5 #v13:5e-6, #v12 1e-8
    'Intrusive igneous rock': 1e-8 * m_s_to_m_month_conversion * k_scalar, #v13: 1e-7

    'Mostly sandstone': 5e-7 * m_s_to_m_month_conversion * k_scalar,
    'Sedimentary material': 1e-6 * m_s_to_m_month_conversion * k_scalar,
    'Eolian sediment': 1e-4 * m_s_to_m_month_conversion * k_scalar,
    'Clastic sediment': 1e-5 * m_s_to_m_month_conversion * k_scalar,
    'Debris flows, landslides, and other localized mass-movement sediment': 1e-5 * m_s_to_m_month_conversion * k_scalar,
    'Peat and muck': 1e-5 * m_s_to_m_month_conversion * k_scalar,
    'Playa sediment': 1e-9 * m_s_to_m_month_conversion * k_scalar,
    'Metamorphic rock': 1e-9 * m_s_to_m_month_conversion * k_scalar,
    'Felsic-composition pyroclastic flows': 1e-6 * m_s_to_m_month_conversion * k_scalar,
    'Coarse-grained intrusive igneous rock': 1e-7 * m_s_to_m_month_conversion * k_scalar,
    'Coarse-grained, felsic-composition intrusive igneous rock': 1e-7 * m_s_to_m_month_conversion * k_scalar,
    'Coarse-grained, mafic-composition intrusive igneous rock': 1e-7 * m_s_to_m_month_conversion * k_scalar,
    'Lacustrine sediment': 1e-9 * m_s_to_m_month_conversion * k_scalar,
    'Mostly carbonate rock': 1e-5 * m_s_to_m_month_conversion * k_scalar,
    'Sedimentary and extrusive igneous material': 1e-6 * m_s_to_m_month_conversion * k_scalar,
    'Water or ice': 1e-9 * m_s_to_m_month_conversion * k_scalar
}

def assign_k_value(feature):
    # Get the text description
    rock_type = feature.get('GeoMateria')
    
    # Earth Engine Dictionaries allow looking up values by string keys
    # We wrap the python dict into an ee.Dictionary
    gee_k_lookup = ee.Dictionary(k_values)
    
    # Get the value (Default to 0.0 if rock type not found)
    k_val = gee_k_lookup.get(rock_type, 0.0)
    
    # Set the new numeric property
    return feature.set('K_m_month', k_val)

# Map over the collection
geomaterials_with_k = geomaterials.map(assign_k_value)

# variable_mask = geomaterials_with_k.reduceToImage(['K_m_month'], ee.Reducer.first()).setDefaultProjection(crs='EPSG:4326', scale=100)
# variable_mask = variable_mask.gte(0.9e-4*m_s_to_m_month_conversion) # Mask for high K values (e.g., Alluvial Sediment, Eolian Sediment)

# Now you can rasterize 'K_m_s' directly if needed
k_raster = geomaterials_with_k.reduceToImage(['K_m_month'], ee.Reducer.first()).setDefaultProjection(crs='EPSG:4326', scale=100).reduceResolution(
    reducer=ee.Reducer.mean(),
    maxPixels=1024
)

# k_raster = ee.Image('projects/ut-gee-ugs-bsf-dev/assets/Utah_USGS_NGMD_Geomaterials_GeoK_m_per_month_1000m')

In [36]:
variable_keys = [
    'Alluvial sediment',
    'Eolian sediment',
    'Clastic sediment',
    'Glacial till',
    'Sedimentary material',
    'Debris flows, landslides, and other localized mass-movement sediment',
    'Peat and muck' # Included: Peat in a valley is likely saturated/discharging, so scaling it to 0 (impermeable) is correct for Recharge.
]

# Create a simple dictionary: { 'Alluvial sediment': 1, 'Eolian...': 1, ... }
variable_dict = {key: 1 for key in variable_keys}

# 2. MAP FUNCTION TO SET THE "IS_VARIABLE" FLAG
def assign_variable_flag(feature):
    # Fix the typo from previous snippets ('GeoMaterial' instead of 'GeoMateria')
    rock_type = feature.get('GeoMateria')
    
    # Wrap our dict for GEE
    gee_var_lookup = ee.Dictionary(variable_dict)
    
    # Look up the rock type. If it's in our list, get 1. If not, default to 0.
    is_variable = gee_var_lookup.get(rock_type, 0)
    
    # Set the flag on the feature
    return feature.set('is_variable', is_variable)

# Apply the flag to the collection
geomaterials_flagged = geomaterials_with_k.map(assign_variable_flag)

# 3. RASTERIZE THE FLAG (Crisp 0 or 1)
# reduceToImage paints the 'is_variable' property.
# We force it to .eq(1) to ensure it is a boolean mask.
variable_mask = geomaterials_flagged.reduceToImage(['is_variable'], ee.Reducer.first()).eq(1).rename('variable_mask')

Creating a KSat scalar raster by combining ruggedness data and elevation data, such that higher elevations that are rugged will have a larger scalar value and lower elevations that are flat will have smaller scalar values. Small scalar values will more significantly reduce the KSat value. This will lower the KSat by an order of magnitude for mountanous regions and multiple orders of magnitude for flat, lowland regions. 

In [7]:
elevation_threshold_low = 6500 #7500 #6500 #6000 #5000
elevation_threshold_high = 8000 #9000 #8500 #8000 #7500

# Predictor for valley floors vs mountain systems. Normalized such that 0 is high elevation (mountainous) and 1 is low elevation (valley).
valley_elev_factor = dem.clamp(elevation_threshold_low, elevation_threshold_high)\
                    .subtract(elevation_threshold_low).divide(elevation_threshold_high - elevation_threshold_low)\
                    .multiply(-1).add(1).rename('Valley_Elevation_Factor')

# Predictor for flat valley floors vs rough mountainous terrain. Normalized such that 1 is flat and 0 is rough.
rug_threshold_flat = 10
rug_threshold_rough = 50
valley_flat_factor = ruggedness.clamp(rug_threshold_flat, rug_threshold_rough)\
                    .subtract(rug_threshold_flat).divide(rug_threshold_rough - rug_threshold_flat)\
                    .multiply(-1).add(1).rename('Valley_Flat_Factor')

valley_intensity = valley_elev_factor.multiply(valley_flat_factor).rename('Valley_Intensity')

k_mountain = 1 #0.005 #0.001 #0.1
k_valley = 0.00001 #0.000005 #0.00001 #0.001

final_geok_scalar = valley_intensity.multiply(k_valley - k_mountain).add(k_mountain).rename('GeoK_Scalar') #.reduceResolution(reducer=ee.Reducer.mean(), maxPixels=1024).reproject(crs='EPSG:32612', scale=1000)

k_raster_scaled = k_raster.where(variable_mask, k_raster.multiply(final_geok_scalar)).reduceResolution(
    reducer=ee.Reducer.mean(),
    maxPixels=1024
)

# final_geok_scalar_asset = ee.Image('projects/ut-gee-ugs-bsf-dev/assets/Utah_GeoK_Scalar_1000m')

NameError: name 'variable_mask' is not defined

#### Handling GeoK scaling exploration using assets

In [40]:
# valley_intensity_100m = ee.Image('projects/ut-gee-ugs-bsf-dev/assets/Utah_Valley_Intensity_100m')
# geok_scalar = valley_intensity_100m.multiply(k_valley - k_mountain).add(k_mountain).rename('GeoK_Scalar')
geok_scalar = valley_elev_factor.multiply(k_valley - k_mountain).add(k_mountain).rename('GeoK_Scalar')

geok_raster_100m = ee.Image('projects/ut-gee-ugs-bsf-dev/assets/Utah_USGS_NGMD_Geomaterials_GeoK_m_per_month_100m')
variable_mask_asset = ee.Image('projects/ut-gee-ugs-bsf-dev/assets/Utah_GeoK_Geomaterials_Variable_Mask_100m')
variable_mask = variable_mask_asset.gt(0)
geok_raster_scaled = geok_raster_100m.where(variable_mask, geok_raster_100m.multiply(geok_scalar)).rename('GeoK_Scaled')

geok_difference = geok_raster_scaled.subtract(previous_best_geok).rename('GeoK_Difference')

Export section

In [8]:
task = ee.batch.Export.image.toAsset(
    image=k_raster_scaled,
    description='Export_Utah_Geomaterials_GeoK_1000m',
    assetId='projects/ut-gee-ugs-bsf-dev/assets/Utah_USGS_NGMD_Geomaterials_GeoK_Scaled_m_per_month_1000m',
    region=UT_boundary,
    scale=1000,           # Explicitly force 1000m output
    crs='EPSG:32612',   # Explicitly force UTM Zone 12N
    maxPixels=1e13      # Allow huge export (Utah is big)
)

task.start()

In [21]:
task = ee.batch.Export.image.toAsset(
    image=k_raster,
    description='Export_Utah_Geomaterials_GeoK_100m_v18',
    assetId='projects/ut-gee-ugs-bsf-dev/assets/Utah_USGS_NGMD_Geomaterials_GeoK_m_per_month_100m',
    region=UT_boundary,
    scale=100,           # Explicitly force 100m output
    crs='EPSG:32612',   # Explicitly force UTM Zone 12N
    maxPixels=1e13      # Allow huge export (Utah is big)
)

task.start()

In [37]:
task = ee.batch.Export.image.toAsset(
    image=variable_mask,
    description='Export_Utah_Geomaterials_Variable_Mask_100m',
    assetId='projects/ut-gee-ugs-bsf-dev/assets/Utah_GeoK_Geomaterials_Variable_Mask_100m',
    region=UT_boundary,
    scale=100,           # Explicitly force 100m output
    crs='EPSG:32612',   # Explicitly force UTM Zone 12N
    maxPixels=1e13      # Allow huge export (Utah is big)
)

# task = ee.batch.Export.image.toAsset(
#     image=valley_intensity,
#     description='Export_Utah_Valley_Intensity',
#     assetId='projects/ut-gee-ugs-bsf-dev/assets/Utah_Valley_Intensity_100m',
#     region=UT_boundary,
#     scale=100,           # Explicitly force 100m output
#     crs='EPSG:32612',   # Explicitly force UTM Zone 12N
#     maxPixels=1e13      # Allow huge export (Utah is big)
# )

task.start()

In [ ]:
# k_raster_scaled = k_raster.multiply(final_geok_scalar)



In [45]:
Map = geemap.Map(center=[39.5, -111.5], zoom=7)
# Map.addLayer(dem, {'min': 4000, 'max': 11000, 'palette':get_palette('dem')}, 'Elevation')
# Map.addLayer(slope, {'min': 0, 'max': 60, 'palette':get_palette('jet')}, 'Slope')
# Map.addLayer(slope, {'min': 0, 'max': 60, 'palette':get_palette('inferno')}, 'Slope')
# Map.addLayer(tpi.abs(), {'min': 0, 'max': 50, 'palette':get_palette('inferno')}, 'TPI Abs')
# Map.addLayer(tpi, {'min': -50, 'max': 50, 'palette':get_palette('rdylbu')}, 'TPI')
Map.addLayer(valley_elev_factor, {'min': 0, 'max': 1, 'palette':get_palette('blues')}, 'Valley Elevation Factor')
# Map.addLayer(valley_flat_factor, {'min': 0, 'max': 1, 'palette':get_palette('blues')}, 'Valley Flat Factor')
# Map.addLayer(valley_intensity, {'min': 0, 'max': 1, 'palette':get_palette('blues')}, 'Valley Intensity')
# Map.addLayer(lithology, {'min': 0, 'max': 12, 'palette':get_palette('jet')}, opacity=0.6, name='Lithology')
# Map.addLayer(final_ksat_scalar_asset, {'min': 0.000005, 'max': 0.005, 'palette':get_palette('greens')}, 'KSat Scalar Asset')
# Map.addLayer(final_ksat_scalar, {'min': 0.000005, 'max': 0.005, 'palette':get_palette('greens')}, 'new KSat Scalar')

# Map.addLayer(valley_intensity_100m, {'min': 0, 'max': 1, 'palette':get_palette('blues')}, 'Valley Intensity')
Map.addLayer(geok_scalar, {'min': 0.00001, 'max': 1, 'palette':get_palette('blues')}, 'GeoK Scalar')
Map.addLayer(geok_raster_100m, {'min': 1e-10*m_s_to_m_month_conversion, 'max': 1e-5*m_s_to_m_month_conversion, 'palette': get_palette('inferno')}, 'GeoK Raster 100m')
Map.addLayer(geok_raster_scaled, {'min': 1e-10*m_s_to_m_month_conversion, 'max': 1e-5*m_s_to_m_month_conversion, 'palette': get_palette('inferno')}, 'GeoK Raster Scaled 100m')
Map.addLayer(geok_difference, vis_params={'min':-100, 'max':101, 'palette':['red','white','blue']}, name='GeoK Difference')
# Map.addLayer(k_raster_scaled, {
#     'min': 1e-8*m_s_to_m_month_conversion, 
#     'max': 1e-4*m_s_to_m_month_conversion, 
#     'palette': ['blue', 'cyan', 'green', 'yellow', 'red'] # Low K (Blue) -> High K (Red)
# }, 'Hydraulic Conductivity (m/month)')
# Map.addLayer(final_geok_scalar, {'min': 0.00001, 'max': 1, 'palette':get_palette('greens')}, 'new KSat Scalar')
# Map.addLayer(geok_difference, vis_params={'min':-100, 'max':101, 'palette':['red','white','blue']}, name='GeoK Difference')

# Map.addLayer(dirty_devil_roi, {'color': 'red'}, 'Dirty Devil Gage Drainage Area')
Map.addLayer(duschesne_roi, {'color': 'blue'}, 'Duschesne Gage Drainage Area')
Map.addLayer(price_roi, {'color': 'green'}, 'Price River Gage Drainage Area')
Map.addLayer(sevier_roi, {'color': 'yellow'}, 'Sevier River Gage Drainage Area')
Map.addLayer(uinta_roi, {'color': 'purple'}, 'Uinta River Gage Drainage Area')
Map.addLayer(weber_roi, {'color': 'orange'}, 'Weber River Oakley Gage Drainage Area')


# Map.addLayer(aspect, {'min': 0, 'max': 360}, 'Aspect')
# Map.addLayer(land_cover, {'min': 0, 'max': 95}, 'Land Cover')
# Map.addLayer(crop_types, {'min': 0, 'max': 255}, 'Crop Types')
# Map.addLayer(lithology, {'min': 0, 'max': 12, 'palette':get_palette('jet')}, 'Lithology')
Map

Map(center=[39.5, -111.5], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright'…

In [55]:
col = ee.ImageCollection("projects/sat-io/open-datasets/HiHydroSoilv2_0/ksat")
ksat = col.min().multiply(10).multiply(30.4375).clip(UT_boundary) # convert from cm/day to mm/month
ksat_min = ksat.reduce(ee.Reducer.min())
ksat_max = ksat.reduce(ee.Reducer.max())

rock_palette = [
    "#F0A3FF", "#0075DC", "#993F00", "#4C005C", "#191919", "#005C31", 
    "#2BCE48", "#FFCC99", "#808080", "#94FFB5", "#8F7C00", "#9DCC00", 
    "#C20088", "#003380", "#FFA405", "#FFA8BB", "#426600", "#FF0010", 
    "#5EF1F2", "#00998F", "#E0FF66", "#740AFF", "#990000", "#FFFF80", 
    "#FFFF00", "#FF5005", "#404040"
]
vis_k = {
    'min': -8, # Log10(1e-8)
    'max': -4, # Log10(1e-4)
    'palette': ['blue', 'cyan', 'green', 'yellow', 'red'] # Low K (Blue) -> High K (Red)
}

# We visualize the Log10 of the image to see the contrast

Map = geemap.Map(center=[39.5, -111.5], zoom=7)
Map.addLayer(dem, {'min': 4000, 'max': 11000, 'palette':get_palette('dem')}, 'Elevation')
# Map.addLayer(geomaterials, vis_params={'band':'palette':rock_palette}, name='GeoMaterials')
# Get unique classes to create an order
classes = geomaterials.aggregate_array('GeoMateria').distinct().sort()

def add_style_from_string(feature):
    # Find where this feature's class name sits in the list (0, 1, 2...)
    class_index = classes.indexOf(feature.get('GeoMateria'))
    color = ee.List(rock_palette).get(class_index)
    return feature.set('style', {
        'fillColor': color,   # The fill color (dynamic)
        'color': color,       # The outline color (dynamic)
        'width': 1            # Outline width (static, or make dynamic too)
    })

colored_geomaterials = geomaterials.map(add_style_from_string)

# 3. Visualize using .style()
# 'fillColor' tells GEE to look for the property name 'style_color' inside the feature
vector_viz = colored_geomaterials.style(
    styleProperty='style'                   # Outline width
)

# 4. Add to Map
# Note: vector_viz is now an RGB Image, so we don't need visualization params in addLayer
Map.addLayer(geomaterials, {}, 'GeoMaterials (Vectors)')
Map.addLayer(vector_viz, {}, 'GeoMaterials (Styled)')
# Map.addLayer(geomaterials, {}, 'GeoMaterials (Vector Style)')
# Map.addLayer(k_raster.log10(), vis_k, 'Hydraulic Conductivity (Log Scale)')
# Map.addLayer(k_raster, {
#     'min': 1e-8*m_s_to_m_month_conversion, 
#     'max': 1e-4*m_s_to_m_month_conversion, 
#     'palette': ['blue', 'cyan', 'green', 'yellow', 'red'] # Low K (Blue) -> High K (Red)
# }, 'Hydraulic Conductivity (m/month)')

# Map.addLayer(k_raster, {
#     'min': 0, 
#     'max': 1, 
#     'palette': ['blue', 'cyan', 'green', 'yellow', 'red'] # Low K (Blue) -> High K (Red)
# }, 'Hydraulic Conductivity (m/month)')

Map.addLayer(geok_raster_scaled.log10(), {
    'min': 0, 
    'max': 2, 
    'palette': ['blue', 'cyan', 'green', 'yellow', 'red'] # Low K (Blue) -> High K (Red)
}, 'Hydraulic Conductivity (m/month)')
# Map.addLayer(ksat, {
#     'min': 1e5, 
#     'max': 1e8, 
#     'palette': ['blue', 'cyan', 'green', 'yellow', 'red'] # Low K (Blue) -> High K (Red)
# }, 'ksat (m/month)')

# Map.addLayer(geok_difference, vis_params={'min':-10, 'max':10, 'palette':['red','white','blue']}, name='GeoK Difference')

Map.addLayer(dirty_devil_roi, {'color': 'red'}, 'Dirty Devil Gage Drainage Area')
Map.addLayer(duschesne_roi, {'color': 'blue'}, 'Duschesne Gage Drainage Area')
Map.addLayer(price_roi, {'color': 'green'}, 'Price River Gage Drainage Area')
Map.addLayer(sevier_roi, {'color': 'yellow'}, 'Sevier River Gage Drainage Area')
Map.addLayer(uinta_roi, {'color': 'purple'}, 'Uinta River Gage Drainage Area')
Map.addLayer(weber_roi, {'color': 'orange'}, 'Weber River Oakley Gage Drainage Area')




Map

Map(center=[39.5, -111.5], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright'…